# Module 2 — Recall / Manufacturing Quality Risk (openFDA Enforcement)

**Scenario**: A regulatory affairs / quality team wants a risk-ranked view of
manufacturers based on their FDA recall (enforcement) history — frequency,
severity (recall classification), and recency — plus a check for unusual
spikes in recall volume over time.

**This notebook**:
1. Pulls real FDA drug enforcement (recall) records
2. Aggregates to firm level: recall frequency, Class I (most severe) rate, recency
3. Produces a risk-scored firm watchlist
4. Flags anomalous months in overall recall volume
5. Runs a hypothesis test comparing the top firm's severity rate against the rest
6. Writes a stakeholder summary to `reports/enforcement_summary.md`

**Caveat**: `recall_number` counts *recall actions*, not units or patients affected.
A firm with many small Class III recalls is not necessarily riskier than one with
a single Class I recall — that's why classification (severity) is weighted
alongside frequency rather than used alone.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.data_connectors.openfda_enforcement import fetch_recalls
from src.risk_framework.risk_scoring import RiskScoringEngine
from src.risk_framework.anomaly_detection import rolling_volume_anomalies
from src.risk_framework.stats import two_proportion_chi_square

CATEGORY = "drug"

## 1. Pull recall records

In [2]:
recalls = fetch_recalls(CATEGORY, max_records=3000)
recalls["is_class_1"] = (recalls["classification"] == "Class I").astype(int)
recalls["report_dt"] = pd.to_datetime(recalls["report_date"], format="%Y%m%d", errors="coerce")

print(f"Fetched {len(recalls)} recall records")
recalls["classification"].value_counts()

Fetched 3000 recall records


classification
Class II     2403
Class III     300
Class I       297
Name: count, dtype: int64

## 2. Recall severity classifier (predict P(Class I))

`classI_rate` computed directly per firm (below) is noisy for firms with only
one or two recalls — a single Class I recall makes a firm look 100% "severe"
even if that's not a stable pattern. A complementary approach: train a
classifier on the *recall-level* text (`reason_for_recall`) and metadata
(`state`, `voluntary_mandated`) to predict P(Class I), then use the **mean
predicted probability per firm** as a smoother, more statistically grounded
severity signal.

Class I recalls are ~10% of this dataset — evaluated with 5-fold stratified
cross-validation (a single train/test split on an imbalanced dataset gives an
unstable estimate).

In [3]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

CLF_FEATURES_TEXT = "reason_for_recall"
CLF_FEATURES_CAT = ["state", "voluntary_mandated"]

clf_X = recalls[[CLF_FEATURES_TEXT] + CLF_FEATURES_CAT].fillna("UNKNOWN")
clf_y = recalls["is_class_1"]

severity_pipeline = Pipeline([
    ("preprocess", ColumnTransformer([
        ("text", TfidfVectorizer(max_features=300, stop_words="english"), CLF_FEATURES_TEXT),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CLF_FEATURES_CAT),
    ])),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_preds = cross_val_predict(severity_pipeline, clf_X, clf_y, cv=cv)
cv_probs = cross_val_predict(severity_pipeline, clf_X, clf_y, cv=cv, method="predict_proba")[:, 1]

print(f"Class I prevalence: {clf_y.mean():.1%} ({clf_y.sum()} of {len(clf_y)})")
print(classification_report(clf_y, cv_preds))
cv_roc_auc = roc_auc_score(clf_y, cv_probs)
print(f"ROC-AUC (5-fold cross-validated): {cv_roc_auc:.4f}")

# Refit on the full dataset for the model we persist and use for firm-level scoring
severity_pipeline.fit(clf_X, clf_y)
recalls["severity_risk"] = severity_pipeline.predict_proba(clf_X)[:, 1]

Class I prevalence: 9.9% (297 of 3000)
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      2703
           1       0.61      0.91      0.73       297

    accuracy                           0.93      3000
   macro avg       0.80      0.92      0.85      3000
weighted avg       0.95      0.93      0.94      3000

ROC-AUC (5-fold cross-validated): 0.9731


In [4]:
import json
import joblib
from datetime import datetime, timezone
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "recall_severity_classifier.joblib"
joblib.dump(severity_pipeline, model_path)

metadata = {
    "model": "LogisticRegression (TF-IDF + OneHotEncoder pipeline)",
    "purpose": "Predict P(recall is Class I) from recall text and metadata, for firm-level "
               "severity scoring",
    "category": CATEGORY,
    "text_feature": CLF_FEATURES_TEXT,
    "categorical_features": CLF_FEATURES_CAT,
    "training_rows": len(recalls),
    "class_balance": {"class_1": int(clf_y.sum()), "other": int((~clf_y.astype(bool)).sum())},
    "cv_method": "StratifiedKFold(n_splits=5, shuffle=True, random_state=42)",
    "cv_roc_auc": float(cv_roc_auc),
    "cv_classification_report": classification_report(clf_y, cv_preds, output_dict=True),
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "caveats": [
        "Class I recalls are ~10% of the dataset; metrics are 5-fold cross-validated "
        "out-of-fold predictions, not a single train/test split, to avoid an unstable "
        "estimate from a small positive class.",
        "Trained on recall *text and metadata*, not outcomes — it predicts how 'Class-I-like' "
        "a recall reads, which correlates with but does not guarantee the regulator's actual "
        "classification for novel cases.",
    ],
}
with open(models_dir / "recall_severity_classifier.meta.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to {model_path.resolve()}")
print(f"Saved metadata to {(models_dir / 'recall_severity_classifier.meta.json').resolve()}")

Saved model to C:\Users\allif\Downloads\Risk Modelling\models\recall_severity_classifier.joblib
Saved metadata to C:\Users\allif\Downloads\Risk Modelling\models\recall_severity_classifier.meta.json


## 3. Firm-level aggregation and risk scoring

Four normalised (0-1) factors per firm:
- **freq_norm**: number of recalls relative to the firm with the most
- **classI_rate**: proportion of this firm's recalls that were Class I (most severe) — raw rate, noisy for low-volume firms
- **severity_risk**: mean predicted P(Class I) from the severity classifier above — a smoother signal that doesn't swing to 0%/100% on one or two recalls
- **recency_norm**: how recent the firm's most recent recall is, relative to the
  oldest-to-newest range in the data

In [5]:
firm = recalls.groupby("recalling_firm").agg(
    n_recalls=("recall_number", "count"),
    classI_rate=("is_class_1", "mean"),
    severity_risk=("severity_risk", "mean"),
    last_recall=("report_dt", "max"),
).reset_index()

firm["freq_norm"] = (firm["n_recalls"] / firm["n_recalls"].max()).clip(upper=1.0)

date_range_days = (recalls["report_dt"].max() - recalls["report_dt"].min()).days
days_since_last = (recalls["report_dt"].max() - firm["last_recall"]).dt.days
firm["recency_norm"] = (1 - (days_since_last / date_range_days)).clip(lower=0, upper=1)

engine = RiskScoringEngine(weights={
    "freq_norm": 0.35, "severity_risk": 0.35, "classI_rate": 0.1, "recency_norm": 0.2
})
scored_firms = engine.score_dataframe(firm).sort_values("risk_score", ascending=False)
watchlist = scored_firms[["recalling_firm", "n_recalls", "classI_rate", "severity_risk", "risk_score", "risk_band"]]
watchlist.head(10)

,recalling_firm,n_recalls,classI_rate,severity_risk,risk_score,risk_band
361,King Bio Inc.,78,1.0,0.985998,0.8296,Critical
284,"Green Lumber Holdings, LLC",2,1.0,0.978354,0.6462,High
626,Supercore Products Group Inc.,1,1.0,0.981404,0.6243,High
548,Pyramids Wholesale Inc.,2,1.0,0.975816,0.6197,High
223,FA Online Inc,1,1.0,0.978740,0.6170,High
419,"Medinatura New Mexico, inc.",1,1.0,0.905091,0.6158,High
676,Umary-USA.com,1,1.0,0.926372,0.6152,High
701,Volt Candy Wholesale Club,1,1.0,0.984510,0.6025,High
533,Pink Toyz,1,1.0,0.984822,0.5902,High
630,TCP HRB ACQUISITON LLC,2,1.0,0.901655,0.5890,High


## 4. Recall-volume anomalies over time

In [6]:
volume = rolling_volume_anomalies(recalls, date_col="report_date", freq="M", threshold=2.5)
anomalous_months = volume[volume["is_anomaly"]]
print(f"{len(anomalous_months)} anomalous month(s) out of {len(volume)}")
anomalous_months

5 anomalous month(s) out of 168


,count,is_anomaly,zscore
report_date,,,
2014-01-01,106,True,5.131639
2015-04-01,107,True,5.189858
2015-09-01,84,True,3.850808
2016-07-01,70,True,3.035735
2019-03-01,89,True,4.141906


## 5. Is the top firm's severity rate statistically different from the rest?

In [7]:
top_firm_name = scored_firms.iloc[0]["recalling_firm"]
top_firm_recalls = recalls[recalls["recalling_firm"] == top_firm_name]
other_recalls = recalls[recalls["recalling_firm"] != top_firm_name]

severity_test = two_proportion_chi_square(
    success_a=int(top_firm_recalls["is_class_1"].sum()), total_a=len(top_firm_recalls),
    success_b=int(other_recalls["is_class_1"].sum()), total_b=len(other_recalls),
)
print(top_firm_name, severity_test)

King Bio Inc. TestResult(test_name='chi_square_2proportion', statistic=718.4945689026257, p_value=2.8445441512063633e-158, significant_at_05=np.True_)


## 6. Save processed outputs

Persist the scored firm watchlist to `data/processed/enforcement/` for downstream
use (e.g. the Streamlit dashboard), alongside a metadata sidecar.

In [8]:
processed_dir = Path("../data/processed/enforcement")
processed_dir.mkdir(parents=True, exist_ok=True)

scored_firms.to_csv(processed_dir / "firm_watchlist.csv", index=False)
volume.to_csv(processed_dir / "recall_volume.csv")

with open(processed_dir / "processed.meta.json", "w", encoding="utf-8") as f:
    json.dump({
        "category": CATEGORY,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "files": {
            "firm_watchlist.csv": f"{len(scored_firms)} firms, risk-scored",
            "recall_volume.csv": f"{len(volume)} monthly recall-volume rows with anomaly flags",
        },
    }, f, indent=2)

print(f"Wrote {len(scored_firms)} firm rows and {len(volume)} volume rows to {processed_dir.resolve()}")

Wrote 740 firm rows and 168 volume rows to C:\Users\allif\Downloads\Risk Modelling\data\processed\enforcement


## 7. Write stakeholder summary

In [9]:
from datetime import datetime, timezone
from pathlib import Path

top5 = scored_firms.head(5)
n_critical = (scored_firms["risk_band"] == "Critical").sum()
n_high = (scored_firms["risk_band"] == "High").sum()

lines = []
lines.append("# Manufacturing Recall Risk Watchlist\n")
lines.append(f"_Generated {datetime.now(timezone.utc).date()} from {len(recalls):,} openFDA "
              f"{CATEGORY} enforcement (recall) records covering "
              f"{recalls['report_dt'].min().date()} to {recalls['report_dt'].max().date()}._\n")
lines.append("## What this means\n")
lines.append(f"Across {firm.shape[0]:,} firms with at least one recall in this dataset, "
              f"{n_critical} firm(s) are flagged **Critical** and {n_high} firm(s) **High** risk "
              f"based on how often they recall product, how severe those recalls are (Class I = "
              f"most severe, both as a raw rate and via a severity classifier), and how recent "
              f"their most recent recall is.\n")
lines.append("## Top-risk firms\n")
lines.append("| Firm | Recalls | Class I rate | Severity score (model) | Risk band |")
lines.append("|---|---|---|---|---|")
for _, r in top5.iterrows():
    lines.append(f"| {r['recalling_firm']} | {r['n_recalls']} | "
                  f"{r['classI_rate']:.0%} | {r['severity_risk']:.0%} | {r['risk_band']} |")
lines.append("")
lines.append("## Severity classifier\n")
lines.append(f"A text-based classifier (TF-IDF on `reason_for_recall` + firm location/voluntary "
              f"flag, logistic regression) predicts P(Class I) per recall, achieving a "
              f"5-fold cross-validated ROC-AUC of **{cv_roc_auc:.2f}** on {len(recalls):,} "
              f"recalls ({clf_y.sum()} Class I, {clf_y.mean():.1%} prevalence). The mean "
              f"predicted probability per firm (`severity_risk`) is used alongside the raw "
              f"Class I rate in the risk score above, since the raw rate alone is unstable for "
              f"firms with only one or two recalls. Model saved to "
              f"`models/recall_severity_classifier.joblib`.\n")
lines.append("## Severity comparison\n")
sig_text = "is" if severity_test.significant_at_05 else "is not"
lines.append(f"The top-ranked firm's Class I recall rate {sig_text} statistically different "
              f"from all other firms combined (chi-square p={severity_test.p_value:.4f}).\n")
lines.append("## Reporting-volume anomalies\n")
lines.append(f"{len(anomalous_months)} month(s) in this dataset had recall volumes more than "
              "2.5 standard deviations from the mean — worth checking against known industry "
              "events (e.g. contamination outbreaks, supply-chain issues, new FDA enforcement "
              "priorities).\n")
lines.append("## Important caveats\n")
lines.append("- A recall *count* does not reflect units shipped, patients affected, or company "
              "size — a large manufacturer will naturally have more recalls than a small one.\n")
lines.append("- Firms with only one or two recalls can show a 100% raw Class I rate purely by "
              "chance — the model-based `severity_risk` score is included specifically to "
              "reduce (not eliminate) this small-sample instability.\n")
lines.append("- This is a single snapshot; trend direction (improving vs. worsening) requires "
              "re-running this analysis over time.\n")
lines.append("## Recommended next steps\n")
lines.append("1. Quality/regulatory team to review the Critical/High firms' `reason_for_recall` "
              "text for common root causes (e.g. contamination, labeling, potency).\n")
lines.append("2. Normalise by firm size/market share if available, to avoid penalising large "
              "manufacturers purely for volume.\n")
lines.append("3. Track risk bands over successive quarters to identify firms trending toward "
              "higher risk.\n")

report_path = Path("../reports/enforcement_summary.md")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote {report_path.resolve()}")

Wrote C:\Users\allif\Downloads\Risk Modelling\reports\enforcement_summary.md
